# SFINCS Demo

In this demo, we will practice building and creating a SFINCS model for the NJ coast from Sandy Hook to Asbury Park during Hurricane Sandy. The northern half (Sandy Hook spit + Sandy Hook Bay) is included so the offshore boundary can carry the alongshore gradient observed during Sandy, and so the NOAA Sandy Hook gauge falls inside the domain for validation.

> 📊 **Interactive (hvplot) maps don't render on github.com** — it strips the JavaScript they need. View this notebook on **[nbviewer](https://nbviewer.org/github/tyfolino/nj_sandy_sfincs/blob/master/notebooks/sfincs-asbury-sandy.ipynb)** for the interactive flood map. (The static Matplotlib flood map renders fine directly on GitHub.)

## Overview & modeling choices

A **hindcast of Hurricane Sandy (28–31 Oct 2012)** flooding on the NJ coast from Sandy Hook to Asbury Park, in three phases: **Phase 1** (slow, one-time geometry build), **Phase 2** (fast forcing + run), **Phase 3** (flood maps + validation).

### Modeling choices at a glance

| Aspect | Choice |
|---|---|
| Event / window | Hurricane Sandy, 2012-10-28 → 10-31 UTC (landfall ≈ 10-29 23:30) |
| Domain | NJ coast, Sandy Hook → Asbury Park (~40.15–40.50 °N) |
| Grid | 50 m rotated UTM 18N, 8 subgrid pixels (~3 m effective) |
| Elevation | 4-tier merge, **pre-Sandy 2010 USACE NCMP on top** (captures the pre-replenishment dune state) |
| Active / outflow | `zb ≥ −10 m` active; `mask=3` outflow on lateral lows |
| Water-level boundary | NOAA CO-OPS gauges (Battery, Atlantic City, Cape May), `buffer=100 km`, 2 alongshore support points. Sandy Hook excluded (failed mid-storm) |
| Wave setup | **Stockdon (2006)** parametric at the boundary, `β_f=0.05`, from ERA5 waves per support point — adds setup, not runup |
| Wind + pressure | ERA5 hourly |
| Rainfall | NOAA AORC v1.1 (~1 km hourly), ~34 mm here |
| Discharge | USGS daily-mean (Shark R, Navesink) at the in-domain estuary inflow cells |
| Infiltration | SCS Curve Number (NLCD × SSURGO HSG) — consumes rainfall only |
| Coriolis / advection | both on (lat 40.32°) |
| Solver | SFINCS v2.3.2 (Docker), regular grid + subgrid |
| Validation | Sandy Hook gauge + USGS pre-storm gauges + 1 storm-tide sensor + **31 USGS HWMs** + **FEMA MOTF** extent |

**Next steps on the roadmap:** SnapWave + IG wavemakers on a quadtree rebuild (the real fix for open-coast runup).

## Data Sources

**Elevation / topobathy** (merged top → bottom; later entries only fill NoData):

| Dataset | File | Purpose |
|---|---|---|
| **2010 USACE NCMP topobathy** (NOAA ID 9456) | `data/elevation/usace_nj_2010_topobathy.tif` | 1 m **pre-Sandy** topobathy on dune/beach/nearshore. NAVD88. |
| NOAA CUDEM 1/9″ | `data/elevation/cudem_asbury.tif` | ~3 m fill for inlets + shelf where NCMP has no data. NAVD88. |
| NJ 10-ft LiDAR | `data/elevation/nj_10ft_dem.tif` | Inland (`zmin=0.001` excludes hydroflattened water). NAVD88. |
| GEBCO 2026 | `data/elevation/gebco_nj.tif` | Offshore tail (≥ ~450 m). |

> *Why pre-Sandy data?* Modern DEMs include post-storm beach replenishment and engineered dunes — using them systematically under-predicts overtopping.

**Forcing:**

| Dataset | File | Purpose |
|---|---|---|
| NOAA CO-OPS water levels (4 gauges) | `data/gtsm/noaa_sandy_nj.nc` | Hourly NAVD88; Sandy Hook excluded (failed mid-storm) and lives in the validation file. |
| ERA5 winds + MSLP | `data/era5/era5_nj_sandy_2012_10_28_31.nc` | Hourly, Oct 28–31 2012. |
| ERA5 wave field | `data/waves/era5_waves_nj.nc` | Hs, Tp per boundary support point for the Stockdon setup. |
| NOAA AORC rainfall | `data/precip/aorc_sandy_nj.nc` | ~1 km hourly QPE (covers 2012 unlike MRMS). |
| USGS river discharge | `data/discharge/usgs_sandy_discharge.nc` | Daily-mean for Shark R + Navesink/Shrewsbury. |

**Roughness + infiltration:**

| Dataset | File | Purpose |
|---|---|---|
| NLCD 2012 | `data/roughness/nlcd_2012.tif` | Manning's-n reclass via `data/roughness/NLCD_CONUS_mapping.csv` (Bunya/Atkinson Atlantic-coast values, NJ-tuned at classes 23/24) + the LULC half of CN. |
| SCS Curve Number | `data/infiltration/cn_nj.nc` | NLCD × SSURGO HSG → infiltration grid. |

**Validation:**

| Dataset | File | Use |
|---|---|---|
| NOAA Sandy Hook (pre-failure) | `data/gtsm/noaa_sandy_validation.nc` | Temporal check up to gauge failure. |
| USGS in-domain tidal gauges | `data/gtsm/usgs_sandy_tidal_nj.nc` | Pre-storm tide range/phase at Shark R + Shrewsbury. |
| USGS storm-tide sensor | `data/gtsm/sandy_storm_tide_nj.nc` | The one in-domain record that survived the peak (open coast 40.37). |
| USGS HWMs (31, in-domain) | `data/validation/sandy_hwms.geojson` | Spatial peak validation. |
| FEMA MOTF surge extent | `data/validation/sandy_motf_extent.tif` | Spatial flood-extent consistency (CSI/POD/FAR). |

In [ ]:
# Imports
import os
import subprocess
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import hvplot.pandas
import hvplot.xarray
import matplotlib.pyplot as plt
import pandas as pd
import rioxarray
import xarray as xr
from hydromt._utils import log
from hydromt_sfincs import DATADIR, SfincsModel, utils
from shapely.geometry import Point



## Phase 1 — Static build

Everything here depends only on **geometry, topography, and roughness** — grid, elevation, mask, observation points, and the subgrid tables. It's the expensive part (the subgrid build spikes to ~13 GB RAM), but it's a **one-time cost**: the outputs are written to disk and SFINCS just reads them.

**Re-run Phase 1 only when one of these changes:** region/resolution, the elevation datasets or merge order, the roughness datasets or reclass table, `nr_subgrid_pixels`, or the mask `zmin`/boundary logic.

If you're only changing **forcing** (water level, wind, pressure, timing), skip straight to Phase 2 — it reopens this model from disk and never touches the subgrid build.

### 1. Initialize SfincsModel, set data library, and output folder

In [ ]:
model_root = "../model"

log.initialize_logging()
log.set_log_level(log_level=30)  # Only errors and critical

# Add file handler to log to a file
log_file = Path(model_root) / "hydromt_sfincs.log"
logger = log._add_filehandler(log_file)

# Add data catalogs
data_libs = ["../data/data_catalog.yml"]

# Create model
sf = SfincsModel(
    data_libs=data_libs, root=model_root, mode="w+", write_gis=True
)


### 2. Specify grid

In [ ]:
# Create grid from region.  50 m (was 100 m) so the 1 m topobathy data
# is sampled into a grid that can actually resolve the barrier-island
# dune line; subgrid pixels below give a ~3 m effective resolution.
sf.grid.create_from_region(
    region={"geom": "../data/region.geojson"},
    res=50,
    rotated=True,
    crs="utm",
)

# Show model grid outline
_ = sf.plot_basemap(
    variable="grid",
    plot_region=True,
    bmap="sat",
    zoomlevel=10,
    grid_kwargs={"linewidth": 0.3},
)


### 3. Add elevation

In [ ]:
elevation_list = [
    {"elevation": "usace_nj_2010"},                # 1 m pre-Sandy topobathy: dunes, beach, surf zone
    {"elevation": "cudem_nj"},                     # 3 m fill: inlet channels + shelf
    {"elevation": "nj_10ft_dem", "zmin": 0.001},   # 3 m fill: inland (drop hydroflat water)
    {"elevation": "gebco_nj"},                     # 450 m offshore tail
]

sf.elevation.create(elevation_list=elevation_list, buffer_cells=1)

# Plot the merged topobathy (colour limits set between -25 m and +10 m to show NJ shelf)
_ = sf.plot_basemap(
    variable="dep",
    plot_region=True,
    bmap="sat",
    zoomlevel=10,
)

### 4. Make mask of activate and inactive cells

In [ ]:
# All cells with elevation >= -10 m are considered active
# The NJ shelf is shallow, so -10 m captures it without going too far offshore
sf.mask.create_active(zmin=-10)

_ = sf.plot_basemap(
    variable="mask",
    plot_region=False,
    plot_bounds=True,
    bmap="sat",
    zoomlevel=10,
)

### 5. Update mask with boundary cells

Two boundary types are set on the perimeter of the active mask:
- **Water level (mask=2)** on the deep-ocean edge (`zmax=-1`) — where NOAA CO-OPS forcing is applied.
- **Outflow (mask=3)** on the lateral/inland edges at low elevation (`-1 ≤ zb ≤ 2 m`) — lets surge that propagates into back-bay channels (Shark River, Sandy Hook Bay) drain out of the domain instead of piling up against the model edge.

In [ ]:
# Mark offshore edge cells (depth > 1 m) as water level boundary cells
sf.mask.create_boundary(
    btype="waterlevel",
    zmax=-1,
    reset_bounds=True,
)

_ = sf.plot_basemap(
    variable="mask",
    plot_region=False,
    plot_bounds=False,
    bmap="sat",
    zoomlevel=10,
)

In [ ]:
# Add outflow cells (mask=3) on the lateral/inland edges of the active mask.
# zmin/zmax restricts to channels/back-bay cells; high dune cells stay as
# regular interior land (no outflow needed). reset_bounds=False preserves
# the offshore water-level boundary set above.
sf.mask.create_boundary(btype="outflow", zmin=-1, zmax=2, reset_bounds=False)

_ = sf.plot_basemap(
    variable="mask",
    plot_region=False,
    plot_bounds=False,
    bmap="sat",
    zoomlevel=10,
)

### 6. Add observation points

In [ ]:
# Project obs points + the Sandy Hook NOAA gauge for validation. The Sandy
# Hook 8531680 gauge falls inside the domain — modeled zs(t) here vs the
# observed record (valid up to the gauge failure ~23:00 UTC 2012-10-29) is the
# cleanest single-point temporal check; the USGS HWMs (Phase 3) add the
# spatial check across the domain.
obs_gdf = gpd.read_file("../data/obs.geojson")
sandy_hook = gpd.GeoDataFrame(
    {"name": ["sandy_hook_gauge"]},
    geometry=[Point(-74.0091, 40.4669)],  # NOAA 8531680
    crs="EPSG:4326",
)
obs_all = gpd.GeoDataFrame(
    pd.concat([obs_gdf, sandy_hook], ignore_index=True),
    crs="EPSG:4326",
)
sf.observation_points.create(locations=obs_all, merge=False)

_ = sf.plot_basemap(
    variable="dep",
    plot_geoms=True,
    plot_bounds=True,
    bmap="sat",
    zoomlevel=12,
)

### 7. Make subgrid derived tables

Subgrid tables bake elevation and roughness into per-cell lookup tables at higher effective resolution. SFINCS uses these directly and **ignores any standalone manningfile** when subgrid is active, so we set roughness here rather than as a separate model layer.

In [ ]:
# NLCD → Manning reclass table (Bunya/Atkinson Atlantic-coast values, classes 23/24
# NJ-tuned; see data/roughness/NLCD_CONUS_mapping.README.md for sources + reasoning).
reclass_table = "../data/roughness/NLCD_CONUS_mapping.csv"

elevation_list = [
    {"elevation": "usace_nj_2010"},                # 1 m pre-Sandy topobathy: dunes, beach, surf zone
    {"elevation": "cudem_nj"},                     # 3 m fill: inlet channels + shelf
    {"elevation": "nj_10ft_dem", "zmin": 0.001},   # 3 m fill: inland (drop hydroflat water)
    {"elevation": "gebco_nj"},                     # 450 m offshore tail
]
roughness_list = [{"lulc": "nlcd_2012", "reclass_table": reclass_table}]

# nr_subgrid_pixels=8 (was 16) so the 50 m grid samples ~3 m effective —
# fine enough to capture the dune line from the 1 m USACE topobathy.
sf.subgrid.create(
    elevation_list=elevation_list,
    roughness_list=roughness_list,
    nr_subgrid_pixels=8,
    write_dep_tif=True,
    write_man_tif=True,
)

In [ ]:
# Plot one of the 2D subgrid variables — u_navg is the representative depth for momentum fluxes
_ = sf.plot_basemap(
    variable=sf.subgrid.data["u_navg"],
    plot_bounds=False,
    bmap="sat",
    zoomlevel=10,
)

In [ ]:
# --- End of Phase 1: write the static model to disk ---
# Grid, elevation, mask, subgrid tables, and obs points are now on disk.
# None of this depends on forcing, so you don't need to rebuild it when
# iterating on water level / wind / pressure — Phase 2 reopens from here.
sf.write()

# Free the build-time memory. The data-catalog cache + source DEMs held
# ~10 GB resident after the build; Phase 2 opens a fresh handle and doesn't
# need any of it. (Restarting the kernel here works just as well.)
del sf
import gc; gc.collect()

## Phase 2 — Forcing & run

Forcing-only iteration (water level, wind/pressure, rainfall, discharge). Cheap — no DEM or subgrid rebuild. The entry-point cell reopens the static Phase-1 model in `r+` mode and is safe after a kernel restart.

In [ ]:
# --- Phase 2 entry point: reopen the static model from disk ---
# mode="r+" opens the existing Phase 1 model and allows updating it in place.
# Re-defined here (not just in the Phase 1 init cell) so this works as a
# standalone entry point after a kernel restart.
model_root = "../model"
data_libs = ["../data/data_catalog.yml"]

sf = SfincsModel(model_root, data_libs=data_libs, mode="r+")

### 1. Add water level time-series as forcing

Observed NOAA CO-OPS hourly water levels (NAVD88) at three complete-record gauges (Battery, Atlantic City, Cape May). Sandy Hook (8531680) is **excluded from forcing** — its NaN tail after the mid-storm failure collapses the northern boundary; the Battery anchors that latitude. `merge=False` replaces the stale `r+` forcing; `buffer=100000` reaches Atlantic City so the boundary keeps Sandy's alongshore gradient.

In [ ]:
# Set the simulation period to cover Hurricane Sandy's landfall.
# tstart == tref so the model gets ~24 h of calm tide before Sandy arrives.
# Without that head-start, at the first step the boundary jumps from zsini=0
# to ~1.5 m (Battery is mid-tide on 10-29 00:00) and the model rings for an
# hour before settling — visible as a spike/oscillation in the Sandy Hook
# validation plot at the start of the run.
sf.config.update(
    {
        "tref": datetime(2012, 10, 28),
        "tstart": datetime(2012, 10, 28),
        "tstop": datetime(2012, 10, 31),
        "tspinup": 3600.0,    # smoothly ramp the boundary over the first hour
                              # instead of step-applying it (belt-and-suspenders
                              # on top of the 24 h of pre-storm tide above).
        "coriolis": 1,        # on: 40 N + sustained alongshore Sandy winds.
                              # Expect only a small correction though — with
                              # observed-gauge boundaries the regional Ekman
                              # setup is already in the BC; this just adds the
                              # setup generated within the ~40 km domain.
        "latitude": 40.32,    # domain-center latitude (deg). SFINCS needs this
                              # to compute the Coriolis parameter — without it
                              # the run log reports "Coriolis: no" even with
                              # coriolis=1 set. f-plane is fine at 40 km
                              # (Coriolis varies <5% across the domain).
        "advection": 1,       # advection term in the momentum equations. Already
                              # the hydromt-sfincs default (always written to
                              # sfincs.inp), so the run log has shown "Advection:
                              # yes" all along — set explicitly here for the
                              # record. Matters around the inlet and steep surge
                              # gradients, and is required once waves/IG are
                              # added later (per the Carolinas/Florence paper).
        "dtmapout": 3600.0,   # hourly map snapshots -> creates sfincs_map.nc
        "dtmaxout": 86400.0,  # write max values once per day
        "dthisout": 600.0,    # observation point output every 10 min
    }
)

# Apply observed NOAA CO-OPS water levels to the boundary cells.
# noaa_sandy_nj.nc holds 3 complete-record stations: The Battery (8518750),
# Atlantic City (8534720), Cape May (8536110). Sandy Hook (8531680) is
# deliberately excluded — its gauge failed mid-storm and its NaN tail
# collapses the northern boundary (see noaa_sandy_validation.nc for that
# record, used for validation only).
#
# merge=False is REQUIRED here: in Phase 2 the model is opened mode="r+",
# which loads any existing bnd forcing from disk. The default merge=True
# would append to that stale forcing — and because the old Sandy Hook
# station isn't in the new file, nothing dedupes it away, so it survives
# and write() then skips ("no changes detected"). merge=False replaces.
#
# buffer=100000 m (was 50000): the southern boundary is ~88 km from Atlantic
# City, so a 50 km buffer caught only The Battery and forced the whole
# boundary uniformly. 100 km pulls in Atlantic City too, so the boundary
# keeps Sandy's real alongshore gradient (Battery ~3.4 m north, AC ~1.9 m south).
sf.water_level.create(
    geodataset="noaa_sandy_nj",
    buffer=100000,
    merge=False,
)

### 1b. Add parametric wave setup to the boundary (Stockdon 2006)

A stand-in for SnapWave (not supported on regular grids in `hydromt_sfincs` v2.0.0rc2). Adds Stockdon setup to the boundary water levels, after Parker et al. (2023):

$$\eta_\mathrm{setup}(t) = 0.35 \cdot \beta_f \cdot \sqrt{H_0(t) \cdot L_0(t)}, \quad L_0(t) = \frac{g \, T_p(t)^2}{2\pi}$$

- $H_0, T_p$: ERA5 waves (`era5_waves_nj`), sampled at the nearest valid node to each of the two boundary support points (gives an alongshore N–S gradient).
- $\beta_f = 0.05$: tuned down from the open-coast median (0.079) because the uniform application over-bumped the sheltered Sandy Hook gauge.

Stockdon captures **setup, not runup** — the open-coast runup deficit and any remaining bay-leakage need SnapWave (+ IG wavemakers) on a quadtree rebuild.

In [ ]:
import numpy as np
from pyproj import Transformer

# β_f stays at 0.05 — the minimum-defensible NJ foreshore slope. It is now a
# single global coefficient applied to a *spatially-varying* wave field
# (below), so the alongshore contrast comes from H0/Tp, not from β_f.
BETA_F = 0.05
GRAVITY = 9.81

# --- Spatially-varying Stockdon setup from the ERA5 wave field ----------------
# The earlier version added one uniform buoy-derived setup at every boundary
# point. That over-bumps the sheltered north (Sandy Hook Bay) while under-
# serving the high-energy open coast. The water-level boundary has only TWO
# support points (The Battery ~40.70 N, Atlantic City ~39.36 N) which SFINCS
# interpolates alongshore. We sample the nearest *offshore* ERA5 node to EACH
# support point and add its own Stockdon setup -> a north–south gradient.
wl = sf.water_level.data
bzs = wl["bzs"].load().copy()
pt_dim = next(d for d in bzs.dims if d != "time")   # alongshore support-point dim
bnd_times = wl["time"]

# Support points are stored as a `geometry` variable (shapely Points in the
# model CRS) -> pull x/y, reproject to lon/lat for ERA5.
geom = np.atleast_1d(wl["geometry"].values)
px = np.array([g.x for g in geom], dtype="float64")
py = np.array([g.y for g in geom], dtype="float64")
to_wgs = Transformer.from_crs(sf.crs, 4326, always_xy=True)
lons, lats = to_wgs.transform(px, py)

waves = sf.data_catalog.get_rasterdataset("era5_waves_nj")
hs_all, tp_all = waves["hs"], waves["tp"]

# Flatten to a 1-D list of VALID offshore nodes (swh is NaN over land), so the
# nearest-node search is unambiguous about (y, x) ordering.
vmask = np.isfinite(hs_all).any("time").values            # (y, x)
lon2d, lat2d = np.meshgrid(hs_all["x"].values, hs_all["y"].values)
iy, ix = np.where(vmask)
vlon, vlat = lon2d[vmask], lat2d[vmask]

print(f"β_f = {BETA_F}; per-support-point ERA5 Stockdon setup:")
for i in range(bzs.sizes[pt_dim]):
    k = int(np.argmin((vlon - lons[i]) ** 2 + (vlat - lats[i]) ** 2))
    jy, jx = int(iy[k]), int(ix[k])
    hs = hs_all.isel(y=jy, x=jx)
    tp = tp_all.isel(y=jy, x=jx)
    L0 = GRAVITY * tp ** 2 / (2.0 * np.pi)
    eta = (0.35 * BETA_F * np.sqrt(hs * L0)).fillna(0.0)
    eta_b = eta.interp(time=bnd_times, kwargs={"fill_value": 0.0})
    bzs[{pt_dim: i}] = bzs.isel({pt_dim: i}) + eta_b
    print(f"  pt {i} (lat {lats[i]:.2f}) -> node ({vlon[k]:.1f},{vlat[k]:.1f}) "
          f"hs_peak {float(hs.max()):.1f} m -> setup_peak {float(eta_b.max()):.2f} m")

sf.water_level.data["bzs"] = bzs
print(f"boundary bzs peak after setup: {float(bzs.max()):.2f} m")


### 2. Add ERA5 wind + pressure forcing

ERA5 hourly 10-m winds and MSLP. Wind setup on the NJ shelf was Sandy's dominant surge driver here. With observed-gauge boundaries the inverse-barometer signal is already in the BC, so internal MSLP mainly drives the *internal* pressure gradient.

In [ ]:
# ERA5 file is pre-renamed to hydromt_sfincs conventions
# (wind10_u, wind10_v in m/s; press_msl in Pa; coords time/y/x).
# Both methods clip to the model bbox + time range internally.
sf.wind.create(wind="era5_nj")
sf.pressure.create(press="era5_nj")

### 2b. Add rainfall forcing (NOAA AORC)

Direct precipitation from **NOAA AORC v1.1** (~1 km hourly, obs-grounded; covers 2012 unlike MRMS). ~34 mm total over the window — secondary here, since Sandy was surge-dominated and the record rainfall fell inland. `cumulative_input=True` because `APCP_surface` is mm accumulated per 1 h. `aggregate=False` keeps the field spatially distributed (`netamprfile`).

In [ ]:
# AORC precip is accumulated mm per 1-hour interval -> cumulative_input=True.
# aggregate=False keeps it spatially distributed (writes netamprfile / sfincs_netampr.nc).
sf.precipitation.create(precip="aorc_sandy_nj", cumulative_input=True, aggregate=False)

_pr = sf.precipitation.data
_var = "precip_2d" if "precip_2d" in _pr else list(_pr.data_vars)[0]
print(f"precip [mm/hr]: peak={float(_pr[_var].max()):.2f}  "
      f"domain-mean={float(_pr[_var].mean()):.3f}  "
      f"grid={dict(_pr[_var].sizes)}")

### 2c. Add river discharge (USGS)

Fluvial inflow at the two gauged coastal rivers: **Shark River** (01407705) and **Navesink/Shrewsbury** via Swimming R (01407500). Daily-mean (only resolution archived for these small gauges in 2012), peaks ≈ 3.5 / 7.9 m³/s — minor next to the surge but part of the compound framing. The src points sit at the wet estuary inflow cells, not the upstream gauges. `merge=False` for the usual `r+` reason. Uses `discharge_points.create` (not `rivers.create_river_inflow`), so the mask isn't touched.

In [ ]:
# Daily-mean discharge at 2 domain inflows (Shark River, Navesink). The
# geodataset's point coords place the src cells; merge=False replaces any
# stale src/dis loaded by r+. Writes sfincs.src + sfincs.dis (no mask edit).
sf.discharge_points.create(geodataset="usgs_sandy_discharge", merge=False)

_dis = sf.discharge_points.data
print(f"discharge src points: {_dis.sizes.get('index', _dis.sizes.get('stations'))}  "
      f"peak={float(_dis['dis'].max()):.2f} m3/s")

### 2d. Add infiltration (NRCS Curve Number)

Without infiltration the ~34 mm of rain ponds as a thin film on every interior cell (≈ 2.9 M m³ above the surge reach — an artifact). **SCS Curve Number** = f(NLCD 2012, SSURGO HSG), built by `scripts/build_cn_nj.py`. SFINCS consumes **rainfall only** with SCS, so coastal inundation is unaffected. `antecedent_moisture=None` → CN II (average). Placed in Phase 2 only to skip a costly Phase-1 subgrid rebuild.

In [ ]:
# SCS Curve Number infiltration (static soil property). antecedent_moisture=None
# -> read the 'cn' variable (CN II / average) directly. Writes sfincs.scs + scsfile.
# SCS is a rainfall-loss method: it only removes infiltration from precipitation,
# never from the surge, so coastal inundation depth is unaffected.
sf.infiltration.create_cn(cn="cn_nj", antecedent_moisture=None)

_scs = sf.grid.data["scs"]
print(f"SCS max soil-moisture retention S [inch]: "
      f"mean={float(_scs.where(_scs > 0).mean()):.2f}  max={float(_scs.max()):.2f}  "
      f"(higher S = more infiltration capacity; S=0 over water/impervious)")

### 3. Save out model

Writes the forcing (and updated config) into the model directory alongside the Phase 1 static files.

In [ ]:
# write all model files to disk
sf.write()

# Workaround: hydromt-sfincs (v2.0.0rc2 on disk) silently drops `latitude`
# from sfincs.inp even when set via config.update. We confirmed empirically:
#   - sf.config.get("latitude") returns 40.32 right after the update
#   - but post-write the inp has no latitude line, AND in-memory `latitude`
#     reverts to 0.0 (sf.write() resets something internally)
# Without `latitude`, SFINCS disables Coriolis (log: "Coriolis: no") even
# with `coriolis = 1` set. Patch the inp directly. Idempotent; remove once
# upstream fix lands.
LATITUDE_DEG = 40.32   # keep in sync with the config.update cell above
inp = Path(model_root) / "sfincs.inp"
text = inp.read_text()
print(f"sf.config.get('latitude') at write-time: {sf.config.get('latitude')}")
if "\nlatitude" in text:
    print(f"sfincs.inp already has a latitude line — leaving it alone")
else:
    text = text.replace(
        "coriolis             = 1",
        f"coriolis             = 1\nlatitude             = {LATITUDE_DEG}",
    )
    inp.write_text(text)
    print(f"patched sfincs.inp: added latitude = {LATITUDE_DEG}")

### 4. Run SFINCS (Singularity on HPC / Docker locally)

Runs the model in the official `deltares/sfincs-cpu` container, mounting the model directory at `/data`. The `run_sfincs()` helper auto-detects the runtime: **Singularity** on Amarel (no Docker daemon on HPC; it runs as you, so no root-owned-file cleanup is needed) or **Docker** on a local desktop (with a pre-run cleanup of root-owned `sfincs_map.nc`/`sfincs_his.nc`). Set `SFINCS_SIF` to override the image path (default `../sfincs-cpu.sif`).

For long or production solves on Amarel, submit `sbatch hpc/sfincs_run.slurm <model_dir>` to a compute node instead of running inline -- never solve on the login node.

In [ ]:
log_path = Path(model_root) / "sfincs_log.txt"
model_abs = Path(model_root).resolve()


def run_sfincs(model_abs, log_path):
    """Run the SFINCS solve in the official deltares/sfincs-cpu container.

    Auto-detects the container runtime so the same notebook runs everywhere:
      * Singularity (Amarel / any HPC -- no Docker daemon allowed there)
      * Docker      (local desktop)
    Reads OMP_NUM_THREADS from the environment (falls back to all cores).

    For long / production solves on Amarel, prefer submitting
    `sbatch hpc/sfincs_run.slurm <model_dir>` to a compute node rather than
    running inline here (never solve on the login node).
    """
    import shutil

    threads = os.environ.get("OMP_NUM_THREADS") or str(os.cpu_count() or 1)

    if shutil.which("singularity"):
        sif = Path(os.environ.get("SFINCS_SIF", Path.cwd().parent / "sfincs-cpu.sif")).resolve()
        print(f"Running SFINCS via Singularity ({sif.name}) on {model_abs}  "
              f"[OMP_NUM_THREADS={threads}] ...")
        env = {**os.environ, "OMP_NUM_THREADS": threads,
               "SINGULARITYENV_OMP_NUM_THREADS": threads}
        with open(log_path, "w") as log_file:
            return subprocess.run(
                ["singularity", "run", "--bind", f"{model_abs}:/data",
                 "--pwd", "/data", str(sif)],
                stdout=log_file, stderr=subprocess.STDOUT, env=env,
            )

    if shutil.which("docker"):
        # Docker writes outputs as root; clear stale root-owned files first or
        # SFINCS errors with "NetCDF: Not a valid ID" overwriting them.
        subprocess.run(
            ["docker", "run", "--rm", "-v", f"{model_abs}:/data",
             "--entrypoint", "/bin/sh", "deltares/sfincs-cpu:latest",
             "-c", "rm -f /data/sfincs_map.nc /data/sfincs_his.nc"],
            capture_output=True,
        )
        print(f"Running SFINCS via Docker on {model_abs} ...")
        with open(log_path, "w") as log_file:
            return subprocess.run(
                ["docker", "run", "--rm", "-v", f"{model_abs}:/data",
                 "deltares/sfincs-cpu:latest"],
                stdout=log_file, stderr=subprocess.STDOUT,
            )

    raise RuntimeError("Neither 'singularity' nor 'docker' found on PATH.")


result = run_sfincs(model_abs, log_path)
print(f"Done (return code {result.returncode})")


In [ ]:
# Read the SFINCS log file
log_path = Path(model_root) / "sfincs_log.txt"
if log_path.exists():
    print(log_path.read_text())
else:
    print("No log file found — SFINCS has not run yet.")

In [ ]:
# Check that output files were created
map_nc = Path(model_root) / "sfincs_map.nc"
his_nc = Path(model_root) / "sfincs_his.nc"

print(f"sfincs_map.nc exists: {map_nc.exists()}")
print(f"sfincs_his.nc exists: {his_nc.exists()}")

## Phase 3 — Visualization

Read the SFINCS outputs (`sfincs_map.nc`, `sfincs_his.nc`) and inspect the results: peak water levels at obs points, time series, zone stats, validation against the Sandy Hook gauge, and a downscaled flood map. This phase is independent of Phase 1 / Phase 2 — it opens the model fresh in read-only mode, so it works after a kernel restart without re-running the build.

### 1. Read Model Results

In [ ]:
# Open the model read-only for result inspection. Defined standalone so the
# Visualization section works after a kernel restart (doesn't depend on the
# Phase 1 / Phase 2 build cells still being in scope).
model_root = "../model"
data_libs = ["../data/data_catalog.yml"]
model_abs = Path(model_root).resolve()

mod = SfincsModel(model_root, data_libs=data_libs, mode="r")
mod.output.read()

print("Output variables available:")
list(mod.output.data.keys())

In [ ]:
# Plot the model layout — grid, boundary cells, and observation points
fig, ax = mod.plot_basemap(fn_out=None, bmap="sat", figsize=(9, 7), geom_names=["obs"])

### Validation: modeled vs observed water level at Sandy Hook

The Sandy Hook gauge (8531680) is the one in-domain temporal record — but it failed at 10-29 23:00, *before* Sandy's true peak. Compare the modeled curve to the gauge **only over the overlap window**; the modeled full-run peak (~3.9 m) is what's comparable to the canonical Sandy Hook storm-tide estimate (~3.86 m).

In [ ]:
# Modeled zs at the sandy_hook_gauge obs point vs the observed NOAA record.
# Model output at obs points (relocated from the removed peak-table cell).
point_zs = mod.output.data["point_zs"]   # (time, station)
point_zb = mod.output.data["point_zb"]   # (station,)
names = [n.decode() if isinstance(n, bytes) else str(n)
         for n in point_zs["station_name"].values]

val = xr.open_dataset("../data/gtsm/noaa_sandy_validation.nc")
obs_sh = val["waterlevel"].sel(stations=8531680)

# locate the sandy_hook_gauge obs point (names may carry trailing padding)
i_sh = next(k for k, n in enumerate(names) if "sandy_hook" in n)
mod_sh = point_zs.isel(stations=i_sh)
zb_sh = float(point_zb.isel(stations=i_sh).values)
mod_sh_wet = mod_sh.where(mod_sh - zb_sh > 0.01)  # only where the cell is wet

# peak comparison over the window the gauge actually covers
gauge_end = pd.Timestamp("2012-10-29 23:00")
mod_overlap = mod_sh.sel(time=slice(None, gauge_end))
print(f"observed peak (pre-failure): {float(obs_sh.max()):.2f} m NAVD88")
print(f"modeled  peak (same window): {float(mod_overlap.max()):.2f} m NAVD88")
print(f"modeled  peak (full run):    {float(mod_sh.max()):.2f} m NAVD88")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mod_sh["time"], mod_sh_wet.values, lw=2, label="modeled zs (SFINCS)")
ax.plot(obs_sh["time"], obs_sh.values, "k.-", ms=4, label="observed (NOAA 8531680)")
ax.axvline(gauge_end, color="red", ls=":", alpha=0.6, label="gauge fails (10-29 23:00)")
ax.set_ylabel("Water surface elevation [m NAVD88]")
ax.set_xlabel("Time [UTC]")
ax.set_title("Sandy Hook: modeled vs observed water level")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()

### Diagnostic: is the bias the boundary, or the wave setup?

Subtracting the reconstructed Stockdon setup isolates the still-water solution. If the green (no-setup) curve tracks the observed gauge while the blue (with-setup) rides high, the bias is the parametric *setup*, not the boundary. Sign: model − observed.

In [ ]:
# Diagnostic: how much of the Sandy Hook bias is the parametric wave setup?
# We reconstruct the Stockdon setup near the gauge (same ERA5 field + β_f the
# forcing cell 1b uses) and subtract it from the modeled series. FIRST-ORDER
# estimate: it removes the setup imposed at the boundary, but the bay's response
# to a boundary bump is slightly attenuated, so the green curve mildly OVER-
# subtracts. Read green as ~"the still-water solution the gauge actually sees."
# NOTE: reflects the ERA5 forcing — valid after a Phase-2 re-run with cell 1b.
import numpy as np

BETA_F = 0.05
GRAVITY = 9.81
SH_LON, SH_LAT = -74.0091, 40.4669   # NOAA 8531680 gauge location

waves = xr.open_dataset("../data/waves/era5_waves_nj.nc")
hs_all, tp_all = waves["hs"], waves["tp"]
vmask = np.isfinite(hs_all).any("time").values
lon2d, lat2d = np.meshgrid(hs_all["x"].values, hs_all["y"].values)
iy, ix = np.where(vmask)
vlon, vlat = lon2d[vmask], lat2d[vmask]
k = int(np.argmin((vlon - SH_LON) ** 2 + (vlat - SH_LAT) ** 2))
hs, tp = hs_all.isel(y=int(iy[k]), x=int(ix[k])), tp_all.isel(y=int(iy[k]), x=int(ix[k]))
L0 = GRAVITY * tp ** 2 / (2.0 * np.pi)
eta = (0.35 * BETA_F * np.sqrt(hs * L0)).fillna(0.0)
eta_m = eta.interp(time=mod_sh["time"], kwargs={"fill_value": 0.0})

mod_nosetup = mod_sh - eta_m   # approx no-wave still-water baseline at the gauge

# residuals vs the observed record over the window the gauge covers
obs_t = pd.to_datetime(obs_sh["time"].values)
mod_t = pd.to_datetime(mod_sh["time"].values)
obs_i = np.interp(mod_t.view("i8"), obs_t.view("i8"), obs_sh.values,
                  left=np.nan, right=np.nan)
ov = mod_t <= gauge_end
print(f"peak setup reconstructed near gauge: {float(eta_m.max()):.2f} m")
print(f"mean residual (to gauge fail)  WITH setup: {np.nanmean(mod_sh.values[ov] - obs_i[ov]):+.2f} m")
print(f"mean residual (to gauge fail)   NO  setup: {np.nanmean(mod_nosetup.values[ov] - obs_i[ov]):+.2f} m")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(mod_sh["time"], mod_sh.values, lw=2, color="tab:blue",
        label="modeled (with Stockdon setup)")
ax.plot(mod_sh["time"], mod_nosetup.values, lw=1.8, ls="--", color="tab:green",
        label="modeled − setup (≈ no-wave baseline)")
ax.plot(obs_sh["time"], obs_sh.values, "k.-", ms=4, label="observed (NOAA 8531680)")
ax.axvline(gauge_end, color="red", ls=":", alpha=0.6, label="gauge fails")
ax.set_ylabel("Water surface elevation [m NAVD88]")
ax.set_xlabel("Time [UTC]")
ax.set_title("Sandy Hook: how much of the high bias is the wave setup?")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()


### Validation: pre-storm tides at the two in-domain USGS gauges

Two USGS estuary gauges in NAVD88 — Shark River at Belmar (south) and Shrewsbury at Sea Bright (mid-north back-bay). Both records stop ~10-29 04:00 UTC, before the peak (all in-domain permanent gauges failed mid-storm), so this is a **tidal check** (range / phase), sampled at the nearest grid cell to each gauge.

In [ ]:
# Pre-storm tidal validation at the two in-domain USGS gauges (NAVD88).
# Sampled from the gridded map output (zs) at the nearest cell to each gauge,
# since these locations are not model observation points.
import numpy as np
from pyproj import Transformer

usgs = xr.open_dataset("../data/gtsm/usgs_sandy_tidal_nj.nc")
zs_grid = mod.output.data["zs"]                 # (time, y, x), hourly
xc, yc = zs_grid["xc"].values, zs_grid["yc"].values
to_utm = Transformer.from_crs(4326, mod.crs, always_xy=True)
gauge_label = {1407770: "Shark River @ Belmar (south)",
               1407600: "Shrewsbury @ Sea Bright (back-bay)"}

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for ax, sid in zip(axes, usgs["stations"].values):
    o = usgs["waterlevel"].sel(stations=sid).dropna("time")
    X, Y = to_utm.transform(float(usgs["lon"].sel(stations=sid)),
                            float(usgs["lat"].sel(stations=sid)))
    jy, jx = np.unravel_index(int(np.argmin((xc - X) ** 2 + (yc - Y) ** 2)), xc.shape)
    m = zs_grid.isel(y=jy, x=jx)
    mo = m.sel(time=slice(o["time"].min(), o["time"].max()))   # overlap window
    rng_o = float(o.max() - o.min()); rng_m = float(mo.max() - mo.min())
    print(f"{gauge_label[int(sid)]:34s}: tidal range obs {rng_o:.2f} m vs modeled {rng_m:.2f} m")
    ax.plot(m["time"], m.values, lw=2, color="tab:blue", label="modeled zs (nearest cell)")
    ax.plot(o["time"], o.values, "k.-", ms=3, label="observed USGS")
    ax.set_title(f"{gauge_label[int(sid)]}  —  pre-storm tide (record ends ~10-29 04:00 UTC)")
    ax.set_ylabel("WSE [m NAVD88]"); ax.legend(loc="upper left"); ax.grid(alpha=0.3)
fig.autofmt_xdate(); plt.tight_layout()

# Takeaways (this run): the south gauge tracks tidal phase but rides ~0.5 m high
# (same wave-setup-into-sheltered-water bias seen at Sandy Hook); the back-bay
# Shrewsbury cell is nearly flat — the estuary isn't hydraulically resolved at 50 m.

### Validation: the one gauge that caught the peak — USGS storm-tide sensor

A USGS rapid-deployment storm-tide sensor on the open coast at Monmouth Beach (40.37°N, two co-located units), via the STN Flood Event Viewer (NAVD88, GMT). These are **wave sensors**: raw signal includes wave oscillations, and being mounted ~9 ft NAVD88 they de-water in troughs (we mask the floored samples, so the 30-min mean is valid only near the submerged peak). The two units disagree by ~1.5 m — read as a *bracket*, not a precise number.

In [ ]:
# Open-coast storm-tide sensor (peak-capturing) vs modeled zs.
import numpy as np
from pyproj import Transformer

sst = xr.open_dataset("../data/gtsm/sandy_storm_tide_nj.nc")
zs_grid = mod.output.data["zs"]
xc, yc = zs_grid["xc"].values, zs_grid["yc"].values
to_utm = Transformer.from_crs(4326, mod.crs, always_xy=True)
X, Y = to_utm.transform(float(sst["lon"].isel(stations=0)), float(sst["lat"].isel(stations=0)))
jy, jx = np.unravel_index(int(np.argmin((xc - X) ** 2 + (yc - Y) ** 2)), xc.shape)
m = zs_grid.isel(y=jy, x=jx)

print(f"modeled open-coast peak still-water (zs): {float(m.max()):.2f} m NAVD88")
for sid in sst["stations"].values:
    st = sst["stormtide_m"].sel(stations=sid); wm = sst["wavemax_m"].sel(stations=sid)
    print(f"  sensor {int(sid)}: storm-tide (still-water) peak {float(st.max()):.2f} m | "
          f"wave-crest peak {float(wm.max()):.2f} m")

fig, ax = plt.subplots(figsize=(11, 5))
# wave-crest envelope of the better-submerged unit, for context
wm0 = sst["wavemax_m"].sel(stations=int(sst["stations"][-1])).dropna("time")
ax.fill_between(wm0["time"].values, 0, wm0.values, color="tab:orange", alpha=0.12,
                label="sensor wave-crest envelope (incl. waves)")
ax.plot(m["time"], m.values, lw=2.2, color="tab:blue", label="modeled zs (nearest cell)")
for sid, c in zip(sst["stations"].values, ["k", "dimgray"]):
    st = sst["stormtide_m"].sel(stations=sid).dropna("time")
    ax.plot(st["time"], st.values, ".-", ms=3, color=c,
            label=f"sensor {int(sid)} storm-tide (30-min mean, wet only)")
ax.set_xlim(np.datetime64("2012-10-29T18"), np.datetime64("2012-10-30T12"))
ax.set_ylabel("Water surface elevation [m NAVD88]"); ax.set_xlabel("Time [UTC]")
ax.set_title("Open-coast storm-tide sensor (40.37°N) vs modeled — Sandy peak")
ax.legend(loc="upper right", fontsize=8); ax.grid(alpha=0.3)
fig.autofmt_xdate(); plt.tight_layout()

# Takeaway: the two co-located wave sensors bracket the model's still-water peak
# (~3.5 and ~5.0 m vs modeled ~3.8 m), so they don't cleanly resolve the open-coast
# over/under here — but their wave crests (~5-6 m) match the highest HWMs, confirming
# that tail is wave runup the still-water model omits. The HWMs remain the cleaner
# spatial validation; this is the only in-domain record that survived to the peak.

### 2. Downscale Flood Map

In [ ]:
# Reuse `mod` opened above — no need to open a second SfincsModel
da_zsmax = mod.output.data["zsmax"].max(dim="timemax")

# Load the high-res subgrid DEM (written during build via write_dep_tif=True)
depfile = str(model_abs / "subgrid" / "dep_subgrid.tif")
da_dep = mod.data_catalog.get_rasterdataset(depfile)
print(f"Subgrid DEM shape: {da_dep.shape}, resolution: {da_dep.rio.resolution()}")

# Downscale water level to subgrid resolution, then drop deep-ocean cells
# (dep > -0.5) so the open shelf doesn't dominate the flood-map colour scale.
da_hmax = utils.downscale_floodmap(zsmax=da_zsmax, dep=da_dep, hmin=0.05)
da_hmax = da_hmax.where(da_dep > -0.5)

In [ ]:
fig, ax = mod.plot_basemap(
    fn_out=None,
    figsize=(8, 6),
    variable=da_hmax,
    plot_bounds=False,
    plot_geoms=False,
    bmap="sat",
    zoomlevel=11,
    vmin=0,
    vmax=5.0,
    cbar_kwargs={"shrink": 0.6, "anchor": (0, 0)},
)
ax.set_title(f"SFINCS maximum water depth")

### Validation: USGS High Water Marks (spatial — "more or less than Sandy?")

**31 USGS HWMs** in-domain (peak NAVD88 elevations from the rapid-response survey, STN event 24, downloaded by `scripts/download_sandy_hwms.py`). Each mark is compared to the wettest **genuinely-flooded** model cell within a 50 m radius:

- **50 m radius** — the nearest 6 m pixel often lands on the building/raised lot the mark sits on (reads dry), so we look at the adjacent flooded street/yard.
- **`DEPTH_MIN`** — required, else a thin rain film on high ground gives a spuriously high WSE.
- **`GROUND_CAP`** — also require `dep ≤ obs + 0.5 m` so the search can't grab a dune/structure cell whose own water surface is well above the mark.

Sign: + = model higher. Headline = quality ≤ 2 subset.

In [ ]:
# Compare modeled peak STILL-WATER level vs USGS Sandy High Water Marks.
import numpy as np

DEPTH_MIN  = 0.15   # m; only compare to genuinely flooded cells, not thin rain film.
GROUND_CAP = 0.5    # m; a mark at elevation `obs` can only have been wet by water
                    # from cells whose GROUND sits at/below obs (+tol). Without this
                    # the 50 m search can grab a higher dune/structure cell whose own
                    # water surface is well above the mark -> a spurious over-predict
                    # (it was turning one q3 mark into a +2 m outlier).

hmax_val = utils.downscale_floodmap(zsmax=da_zsmax, dep=da_dep, hmin=0.05)
hwm = gpd.read_file("../data/validation/sandy_hwms.geojson")
hwm = hwm.to_crs(da_dep.rio.crs)

depth, dep_arr, wse = hmax_val.values, da_dep.values, (da_dep + hmax_val).values
if depth.ndim == 3:
    depth, wse, dep_arr = depth[0], wse[0], dep_arr[0]
T = da_dep.rio.transform()
ny, nx = wse.shape
rad = int(round(50 / abs(T.a)))

obs = hwm["elev_m"].values
qual = hwm["quality"].values.astype(float)
mod_wse = np.full(len(obs), np.nan)
for k, (X, Y) in enumerate(zip(hwm.geometry.x.values, hwm.geometry.y.values)):
    col, row = int((X - T.c) / T.a), int((Y - T.f) / T.e)
    if 0 <= row < ny and 0 <= col < nx:
        sl = (slice(max(0, row - rad), row + rad + 1),
              slice(max(0, col - rad), col + rad + 1))
        ws, hh, dd = wse[sl], depth[sl], dep_arr[sl]
        flooded = (hh >= DEPTH_MIN) & (dd <= obs[k] + GROUND_CAP)
        if flooded.any():
            mod_wse[k] = np.nanmax(np.where(flooded, ws, np.nan))

wet = np.isfinite(mod_wse)
resid = mod_wse - obs                  # + = model higher than observed
q2 = qual <= 2

def report(label, m):
    if m.sum() == 0:
        print(f"{label}: (none)"); return
    r = resid[m]
    print(f"{label}: n={int(m.sum()):2d}  mean={r.mean():+.2f}  median={np.median(r):+.2f}  "
          f"RMSE={np.sqrt((r**2).mean()):.2f}  within±0.5 m={np.mean(np.abs(r)<0.5)*100:.0f}%")

print(f"model flooded within 50 m of {wet.sum()}/{len(obs)} HWMs  (+ = model over-predicts)")
report("HEADLINE  quality<=2 ", wet & q2)
report("          quality<=3 ", wet & (qual <= 3))
report("          all wet     ", wet)
print(f"model DRY at {int((~wet).sum())} HWMs — still-water can't reach them "
      f"(runup candidates; see the envelope diagnostic below)")

# Scatter — emphasise the trustworthy q<=2 subset; q3-4 shown hollow.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(obs[wet & ~q2], mod_wse[wet & ~q2], facecolor="none", edgecolor="grey",
           s=55, linewidth=0.8, label="q3-4 (low survey quality)")
sc = ax.scatter(obs[wet & q2], mod_wse[wet & q2], c=qual[wet & q2], cmap="viridis_r",
                s=60, edgecolor="k", linewidth=0.4, vmin=1, vmax=5, label="q1-2 (headline)")
lim = [1.8, 6.0]
ax.plot(lim, lim, "k--", lw=1, label="1:1")
ax.fill_between(lim, [l - 0.5 for l in lim], [l + 0.5 for l in lim], color="grey", alpha=0.15)
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
ax.set_xlabel("Observed HWM [m NAVD88]")
ax.set_ylabel("Modeled still-water WSE [m NAVD88]")
ax.set_title("Modeled still-water vs USGS HWMs")
fig.colorbar(sc, ax=ax, shrink=0.8, label="HWM quality (1=best)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout()


### Validation map: where does the model over/under-predict?

The 31 HWMs on the flood map, colored by residual (model − obs). Red = model higher, blue = model lower, black × = dry in the model. Useful for spotting whether the misses cluster spatially (e.g. behind a back-bay shore) or scatter randomly.

In [ ]:
# HWM residuals on the flood map (reuses hwm, resid, wet from the cell above).
import numpy as np

fig, ax = mod.plot_basemap(
    fn_out=None, figsize=(9, 7), variable=da_hmax,
    plot_bounds=False, plot_geoms=False, bmap="sat", zoomlevel=11,
    vmin=0, vmax=5, cmap="Blues", cbar_kwargs={"shrink": 0.5, "label": "Modeled depth [m]"},
)
hx, hy = hwm.geometry.x.values, hwm.geometry.y.values
sc = ax.scatter(hx[wet], hy[wet], c=resid[wet], cmap="RdBu_r", vmin=-1.5, vmax=1.5,
                s=70, edgecolor="k", linewidth=0.6, zorder=5)
ax.scatter(hx[~wet], hy[~wet], marker="x", color="k", s=70, linewidth=1.6,
           zorder=6, label=f"model dry ({int((~wet).sum())})")
fig.colorbar(sc, ax=ax, shrink=0.5, label="HWM residual: model − obs [m]")
ax.legend(loc="upper right")
ax.set_title("Sandy HWM residuals (red = model over-predicts, blue = under)")
plt.tight_layout()

### Validation: are the under-predicted marks just wave runup?

The still-water model has no swash, which on the open beach during Sandy added meters. So an open-coast HWM should fall **above** the model still-water and **below** the full-runup elevation. We bracket each mark:

- **lower bound** = model still-water reach near the mark
- **upper bound** = still-water + **Stockdon (2006) R2%** ($1.1(\eta + S/2)$, setup + incident/IG swash) from the same ERA5 waves and $\beta_f$

If the marks fall inside, the scatter is surge ↔ runup physics: high open-coast marks ride the runup top (the IG/runup tail that needs SnapWave + IG wavemakers, Phase 3), and reds are the separate sheltered-bay setup-leakage story.

In [ ]:
# Runup envelope: bracket each HWM by [still-water .. still-water + Stockdon R2%].
import numpy as np

BETA_F, GRAVITY = 0.05, 9.81

# (a) surf-zone still-water base near each mark (300 m search ~ the beach toe).
rad_toe = int(round(300 / abs(T.a)))
toe = np.full(len(obs), np.nan)
for k, (X, Y) in enumerate(zip(hwm.geometry.x.values, hwm.geometry.y.values)):
    col, row = int((X - T.c) / T.a), int((Y - T.f) / T.e)
    if 0 <= row < ny and 0 <= col < nx:
        sl = (slice(max(0, row - rad_toe), row + rad_toe + 1),
              slice(max(0, col - rad_toe), col + rad_toe + 1))
        ws, hh = wse[sl], depth[sl]
        fl = hh >= DEPTH_MIN
        if fl.any():
            toe[k] = np.nanmax(np.where(fl, ws, np.nan))

# (b) Stockdon R2% from the ERA5 wave field at each mark (same beta_f as cell 1b).
hwm4 = hwm.to_crs(4326)
mlon, mlat = hwm4.geometry.x.values, hwm4.geometry.y.values
waves = xr.open_dataset("../data/waves/era5_waves_nj.nc")
vmask = np.isfinite(waves["hs"]).any("time").values
lon2d, lat2d = np.meshgrid(waves["x"].values, waves["y"].values)
iy, ix = np.where(vmask)
vlon, vlat = lon2d[vmask], lat2d[vmask]
r2 = np.full(len(obs), np.nan)
for k in range(len(obs)):
    j = int(np.argmin((vlon - mlon[k]) ** 2 + (vlat - mlat[k]) ** 2))
    hs = float(waves["hs"].isel(y=int(iy[j]), x=int(ix[j])).max())
    tp = float(waves["tp"].isel(y=int(iy[j]), x=int(ix[j])).max())
    root = np.sqrt(hs * GRAVITY * tp ** 2 / (2 * np.pi))
    eta = 0.35 * BETA_F * root
    S = np.hypot(0.75 * BETA_F * root, 0.06 * root)   # incident + IG swash
    r2[k] = 1.1 * (eta + S / 2)

lower = np.where(wet, mod_wse, toe)    # best model still-water estimate at the mark
upper = toe + r2                       # still-water + full open-coast runup
within = (obs >= lower - 0.5) & (obs <= upper + 0.1)
print(f"Stockdon R2% runup ~ {np.nanmin(r2):.1f}-{np.nanmax(r2):.1f} m above still water (Sandy peak)")
print(f"HWMs inside [still-water .. +R2%] envelope: {within.sum()}/{len(obs)} ({within.mean()*100:.0f}%)")

order = np.argsort(obs)
xx = np.arange(len(obs))
fig, ax = plt.subplots(figsize=(11, 5))
ax.vlines(xx, lower[order], upper[order], color="tab:blue", alpha=0.35, lw=4,
          label="model range: still-water → +R2% runup")
ax.plot(xx, lower[order], "_", color="tab:blue", ms=10)
ax.plot(xx, obs[order], "ko", ms=5, label="observed HWM")
ins = within[order]
ax.plot(xx[~ins], obs[order][~ins], "rx", ms=9, label="outside envelope")
ax.set_xlabel("HWM (sorted by observed elevation)")
ax.set_ylabel("Elevation [m NAVD88]")
ax.set_title("HWMs bracketed by modeled surge ↔ runup (Stockdon R2%)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout()


### Validation: spatial extent vs FEMA MOTF Sandy surge footprint

Footprint-level check: does the model flood the same *area* the FEMA MOTF Sandy extent does? We rasterise MOTF to the model subgrid, restrict to land cells, and score the standard inundation metrics — **CSI** (hits / (hits + miss + FA)), **POD**, **FAR** — plus a categorical hit / miss / false-alarm map.

> **Caveat:** MOTF is a static, HWM/sensor-interpolated "bathtub" surface — not a hydrodynamic run — and shares provenance with our HWMs. Treat this as an extent **consistency** check, not independent validation.

In [ ]:
# Spatial-extent validation vs FEMA MOTF Sandy surge footprint.
# Both rasters are EPSG:32618, so we sample model cells at MOTF pixel centers in
# pure numpy — no GDAL warping needed (avoids known reproject crashes in this env).
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

DEPTH_MIN = 0.15   # m; match the HWM cell's wet threshold

with rasterio.open("../data/validation/sandy_motf_extent.tif") as r:
    motf = r.read(1); mtf = r.transform; m_nd = r.nodata
mod_t = da_dep.rio.transform()
mh, mw = motf.shape

# Map MOTF pixel centers -> nearest model subgrid pixel index (both UTM 18N).
Xc = mtf.c + (np.arange(mw) + 0.5) * mtf.a
Yc = mtf.f + (np.arange(mh) + 0.5) * mtf.e
mc = np.clip(((Xc - mod_t.c) / mod_t.a).astype(int), 0, da_dep.shape[-1] - 1)
mr = np.clip(((Yc - mod_t.f) / mod_t.e).astype(int), 0, da_dep.shape[-2] - 1)
rr, cc = np.meshgrid(mr, mc, indexing="ij")
def _2d(a): return a[0] if a.ndim == 3 else a
dep_at = _2d(da_dep.values)[rr, cc]
h_at   = _2d(da_hmax.values)[rr, cc]

motf_wet = (motf == 1)
mod_wet  = (h_at >= DEPTH_MIN) & np.isfinite(h_at)
land_in  = (motf != m_nd) & (dep_at > 0.0)         # land cells inside the domain
hits = motf_wet &  mod_wet & land_in
miss = motf_wet & ~mod_wet & land_in
fa   = ~motf_wet &  mod_wet & land_in
nh, nm, nf = int(hits.sum()), int(miss.sum()), int(fa.sum())
PIX = mtf.a * abs(mtf.e) / 1e6     # km^2 per pixel
CSI = nh / (nh + nm + nf)
POD = nh / (nh + nm) if (nh + nm) else 0.0
FAR = nf / (nh + nf) if (nh + nf) else 0.0
motf_land = (motf_wet & land_in).sum() * PIX
mod_land  = (mod_wet  & land_in).sum() * PIX
print(f"MOTF flooded land in domain : {motf_land:.1f} km2")
print(f"Model wet land in domain    : {mod_land:.1f} km2")
print(f"  hits {nh * PIX:5.1f}  miss {nm * PIX:5.1f}  false-alarm {nf * PIX:5.1f}  km2")
print(f"  CSI={CSI:.2f}   POD (hit rate)={POD:.2f}   FAR={FAR:.2f}")

# Categorical difference map: hits / misses / false alarms.
cat = np.zeros_like(motf, dtype="uint8")
cat[hits] = 1; cat[miss] = 2; cat[fa] = 3
cmap = ListedColormap([(1, 1, 1, 0),
                       (0.20, 0.60, 0.30, 1.0),
                       (0.20, 0.40, 0.85, 1.0),
                       (0.85, 0.20, 0.20, 1.0)])
ext = [mtf.c, mtf.c + mw * mtf.a, mtf.f + mh * mtf.e, mtf.f]
mod_ext = [mod_t.c, mod_t.c + da_dep.shape[-1] * mod_t.a,
           mod_t.f + da_dep.shape[-2] * mod_t.e, mod_t.f]

fig, ax = plt.subplots(figsize=(7.5, 9))
ax.set_facecolor("#f4f4f0")
ax.imshow(_2d(da_dep.values), extent=mod_ext, cmap="Greys",
          vmin=-5, vmax=20, alpha=0.45, origin="upper", interpolation="nearest")
ax.imshow(cat, cmap=cmap, vmin=0, vmax=3, extent=ext,
          origin="upper", interpolation="nearest")
ax.set_aspect("equal"); ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
ax.set_xlabel("Easting [m, UTM 18N]"); ax.set_ylabel("Northing [m]")
ax.legend(handles=[
    Patch(color=cmap(1), label=f"hit ({nh * PIX:.1f} km²)"),
    Patch(color=cmap(2), label=f"miss — MOTF wet, model dry ({nm * PIX:.1f} km²)"),
    Patch(color=cmap(3), label=f"false alarm — model wet, MOTF dry ({nf * PIX:.1f} km²)"),
], loc="upper right", fontsize=8, framealpha=0.95)
ax.set_title(f"Modeled flood vs FEMA MOTF Sandy extent  —  "
             f"CSI={CSI:.2f}   POD={POD:.2f}   FAR={FAR:.2f}")
plt.tight_layout()

# Takeaway (this run): misses cluster in the back-bays / inland lows (runup + 50 m
# connectivity limits, same family as Deal Lake & Shrewsbury); false alarms at
# Sandy Hook spit & a few southern spots (the bay setup leakage we saw at the
# gauges). Extent picture matches the HWM residual map.

In [ ]:
# Reload obs points from the written model (gis/obs.geojson) rather than
# relying on `obs_all` from the Phase 1 build cell — keeps the Visualization
# section runnable standalone after a kernel restart.
obs_all = gpd.read_file(model_abs / "gis" / "obs.geojson")

flood_wgs84 = da_hmax.where(da_hmax > 0.05).rio.reproject(
    "EPSG:4326", nodata=float("nan")
)

flood_map = flood_wgs84.hvplot.image(
    x="x", y="y",
    cmap="viridis",
    clim=(0, 5),
    geo=True,
    tiles="EsriImagery",
    alpha=0.75,
    width=900, height=700,
    title="Max flood depth — Sandy (NOAA gauges + ERA5 + Stockdon setup + AORC rain + USGS discharge)",
    clabel="Flood depth [m]",
)

obs_layer = obs_all.hvplot.points(
    geo=True, color="red", size=60,
    hover_cols=["name"],
)

flood_map * obs_layer